In [1]:
#https://gist.github.com/jiahao87/50cec29725824da7ff6dd9314b53c4b3

In [2]:
!pip install ohmeow-blurr -q
!pip install datasets -q
!pip install bert-score -q
!pip install transformers

     |████████████████████████████████| 91 kB 4.2 MB/s 
     |████████████████████████████████| 312 kB 35.9 MB/s 
     |████████████████████████████████| 3.8 MB 44.6 MB/s 
     |████████████████████████████████| 189 kB 48.6 MB/s 
     |████████████████████████████████| 43 kB 1.9 MB/s 
     |████████████████████████████████| 1.2 MB 41.7 MB/s 
     |████████████████████████████████| 55 kB 3.5 MB/s 
     |████████████████████████████████| 67 kB 4.5 MB/s 
     |████████████████████████████████| 895 kB 44.0 MB/s 
     |████████████████████████████████| 6.5 MB 35.3 MB/s 
     |████████████████████████████████| 596 kB 48.7 MB/s 
     |████████████████████████████████| 1.1 MB 37.2 MB/s 
     |████████████████████████████████| 134 kB 40.2 MB/s 
     |████████████████████████████████| 212 kB 18.7 MB/s 
     |████████████████████████████████| 127 kB 31.8 MB/s 
     |████████████████████████████████| 144 kB 40.3 MB/s 
     |████████████████████████████████| 94 kB 2.1 MB/s 
     |██████████████████

In [3]:
import datasets
import pandas as pd
from fastai.text.all import *
from transformers import *

from blurr.data.all import *
from blurr.modeling.all import *

In [4]:
df_news = pd.read_csv('news_summary - train.csv', 
                       engine='python',
                       encoding='unicode_escape') 

In [5]:
df_news = df_news[['Text', 'Summary']]
df_news.head()

,Text,Summary
0,"Motorman Mahendra Prasad called up the railway police control room on Sunday and informed officials that a 15-year-old boy and 25-year-old youth had fallen from a train and were found injured on the tracks between Lower Parel and Elphinston stations. Mohammed Ansari, 25, and Mohammed Nisar, 15, were among the 10 people injured while travelling on Western and Central Railway line on Sunday.Sunday recorded 13 track deaths, the highest on a single day this year. In the past six months, around 1,590 people have been killed while travelling on trains ? 676 died after falling off trains. On an a...","As many as 13 people died while travelling in Mumbai's local trains on Sunday, according to railway data. Almost 1,600 people have been killed while travelling in trains in the past six months, it added. Of the 1,600, 700 lost their lives by falling off the train,147 were killed while crossing railway tracks, and 20 people died of electric shock."
1,"Bored in confines of a sprawling resort in Bengaluru, the Congress flock of 44 MLAs is a frustrated lot now. Following strict orders of the Karnataka Power Minister DK Shivakumar, the MLAs are not allowed to venture out in the city. While the minister is presently in New Delhi to seek further course of action, 30 MLAs have been put in-charge of the discipline. But few defiant MLAs did head out in the city for sightseeing, shopping, to visit temples, much to the dismay of those still under curbs. The MLAs who were allowed to go out are supposedly those, the Congress is confident of not ditc...","The Gujarat Congress MLAs staying in a Bengaluru resort are not allowed to go out, and 30 MLAs from Karnataka have been put in charge of the discipline. The party has planned seminars and programmes for their entertainment inside the resort. These include a lecture on the corruption charges against UPA, and Congress' achievements in 60 years, transparency and accountability."
2,"There is only a 5% chance that the Earth will avoid warming by at least 2C come the end of the century, according to new research that paints a sobering picture of the international effort to stem dangerous climate change.Global trends in the economy, emissions and population growth make it extremely unlikely that the planet will remain below the 2C threshold set out in the Paris climate agreement in 2015, the study states. The Paris accord, signed by 195 countries, commits to holding the average global temperature to ?well below 2C? above pre-industrial levels and sets a more aspirational...","According to a recently published US-based research, there is only a 5% chance that Earth will avoid warming by at least 2°C by 2100, the threshold set in the Paris climate agreement. The research studied global trends in the economy, emissions and population growth, and further stated that there is 1% chance that temperatures will rise by less than 1.5°C."
3,"After a swarm of bees held up a Hyderabad-Pune flight last week for more than one hour, the country aviation regulator said it will investigate how a bee hive was allowed to flourish in the airport area. The flight on July 26 was held up when the ATR?72 - a short-haul aircraft - was besieged by bees. The flight, which was scheduled to take off at around 6.25pm, took off only at around 7.22pm after a repellent spray was used to clear the insects from the apron area. This was the first such incident in the past five years. A Directorate General of Civil Aviation (DGCA) official on Monday sa...","A swarm of bees reportedly attacked a Hyderabad-Pune flight last week and delayed it by almost an hour. The flight was able to take off only after a repellent spray was used to clear the insects from the area. ""We will look into the matter and necessary action will be taken,"" a Directorate General of Civil Aviation official said."
4,"New Delhi, Aug 1 (PTI) Flight Lt Shikha Pandey, a member of the Indian womens cricket team, was today felicitated by Indian Air Force chief 

In [6]:
df_news.shape

(149, 2)

In [7]:
df_news = df_news.dropna()

In [8]:
for col in df_news.columns.tolist():
    df_news[col+"_num_words"] = df_news[col].apply(lambda x: len(x.split()))
df_news.head()

,Text,Summary,Text_num_words,Summary_num_words
0,"Motorman Mahendra Prasad called up the railway police control room on Sunday and informed officials that a 15-year-old boy and 25-year-old youth had fallen from a train and were found injured on the tracks between Lower Parel and Elphinston stations. Mohammed Ansari, 25, and Mohammed Nisar, 15, were among the 10 people injured while travelling on Western and Central Railway line on Sunday.Sunday recorded 13 track deaths, the highest on a single day this year. In the past six months, around 1,590 people have been killed while travelling on trains ? 676 died after falling off trains. On an a...","As many as 13 people died while travelling in Mumbai's local trains on Sunday, according to railway data. Almost 1,600 people have been killed while travelling in trains in the past six months, it added. Of the 1,600, 700 lost their lives by falling off the train,147 were killed while crossing railway tracks, and 20 people died of electric shock.",316,60
1,"Bored in confines of a sprawling resort in Bengaluru, the Congress flock of 44 MLAs is a frustrated lot now. Following strict orders of the Karnataka Power Minister DK Shivakumar, the MLAs are not allowed to venture out in the city. While the minister is presently in New Delhi to seek further course of action, 30 MLAs have been put in-charge of the discipline. But few defiant MLAs did head out in the city for sightseeing, shopping, to visit temples, much to the dismay of those still under curbs. The MLAs who were allowed to go out are supposedly those, the Congress is confident of not ditc...","The Gujarat Congress MLAs staying in a Bengaluru resort are not allowed to go out, and 30 MLAs from Karnataka have been put in charge of the discipline. The party has planned seminars and programmes for their entertainment inside the resort. These include a lecture on the corruption charges against UPA, and Congress' achievements in 60 years, transparency and accountability.",109,60
2,"There is only a 5% chance that the Earth will avoid warming by at least 2C come the end of the century, according to new research that paints a sobering picture of the international effort to stem dangerous climate change.Global trends in the economy, emissions and population growth make it extremely unlikely that the planet will remain below the 2C threshold set out in the Paris climate agreement in 2015, the study states. The Paris accord, signed by 195 countries, commits to holding the average global temperature to ?well below 2C? above pre-industrial levels and sets a more aspirational...","According to a recently published US-based research, there is only a 5% chance that Earth will avoid warming by at least 2°C by 2100, the threshold set in the Paris climate agreement. The research studied global trends in the economy, emissions and population growth, and further stated that there is 1% chance that temperatures will rise by less than 1.5°C.",944,60
3,"After a swarm of bees held up a Hyderabad-Pune flight last week for more than one hour, the country aviation regulator said it will investigate how a bee hive was allowed to flourish in the airport area. The flight on July 26 was held up when the ATR?72 - a short-haul aircraft - was besieged by bees. The flight, which was scheduled to take off at around 6.25pm, took off only at around 7.22pm after a repellent spray was used to clear the insects from the apron area. This was the first such incident in the past five years. A Directorate General of Civil Aviation (DGCA) official on Monday sa...","A swarm of bees reportedly attacked a Hyderabad-Pune flight last week and delayed it by almost an hour. The flight was able to take off only after a repellent spray was used to clear the insects from the area. ""We will look into the matter and necessary action will be taken,"" a Directorate General of Civil Aviation official said.",389,59
4,"New Delhi, Aug 1 (PTI) Flight Lt Shikha Pandey, a member of the Indian womens c

In [9]:
df_news.describe()

,Text_num_words,Summary_num_words
count,147.000000,147.000000
mean,443.877551,58.251701
std,403.011164,2.454639
min,67.000000,50.000000
25%,250.500000,57.000000
50%,357.000000,59.000000
75%,501.500000,60.000000
max,3093.000000,60.000000


In [10]:
# df = df_news.sample(100)

In [11]:
# from datasets import Dataset

# dataset = Dataset.from_pandas(df)

In [12]:
# dataset

In [13]:
pretrained_model_name = "facebook/bart-large-cnn"
hf_arch, hf_config, hf_tokenizer, hf_model = BLURR.get_hf_objects(pretrained_model_name, 
                                                                  model_cls=BartForConditionalGeneration)

hf_arch, type(hf_config), type(hf_tokenizer), type(hf_model)

Downloading:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/878k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.51G [00:00<?, ?B/s]

('bart',
 transformers.models.bart.configuration_bart.BartConfig,
 transformers.models.bart.tokenization_bart_fast.BartTokenizerFast,
 transformers.models.bart.modeling_bart.BartForConditionalGeneration)

In [14]:
text_gen_kwargs = default_text_gen_kwargs(hf_config,
                                          hf_model,
                                          task='summarization')
text_gen_kwargs

{'bad_words_ids': None,
 'bos_token_id': 0,
 'decoder_start_token_id': 2,
 'diversity_penalty': 0.0,
 'do_sample': False,
 'early_stopping': True,
 'encoder_no_repeat_ngram_size': 0,
 'eos_token_id': 2,
 'forced_bos_token_id': 0,
 'forced_eos_token_id': 2,
 'length_penalty': 2.0,
 'max_length': 142,
 'min_length': 56,
 'no_repeat_ngram_size': 3,
 'num_beam_groups': 1,
 'num_beams': 4,
 'num_return_sequences': 1,
 'output_attentions': False,
 'output_hidden_states': False,
 'output_scores': False,
 'pad_token_id': 1,
 'remove_invalid_values': False,
 'repetition_penalty': 1.0,
 'return_dict_in_generate': False,
 'temperature': 1.0,
 'top_k': 50,
 'top_p': 1.0,
 'typical_p': 1.0,
 'use_cache': True}

In [15]:
hf_batch_tfm = HF_Seq2SeqBeforeBatchTransform(hf_arch, hf_config,
                                              hf_tokenizer,
                                              hf_model, 
                                              max_length=400, 
                                              max_tgt_length=70, 
                                              text_gen_kwargs=text_gen_kwargs)

In [16]:
blocks = (HF_Seq2SeqBlock(before_batch_tfm=hf_batch_tfm), noop)

In [17]:
dblock = DataBlock(blocks=blocks,
                   get_x=ColReader('Text'),
                   get_y=ColReader('Summary'),
                   splitter=RandomSplitter())

In [18]:
batch_size=2
dls = dblock.dataloaders(df_news, bs=batch_size)

In [19]:
len(dls.train.items), len(dls.valid.items)

(118, 29)

In [20]:
b = dls.one_batch()
len(b), b[0]['input_ids'].shape, b[1].shape

(2, torch.Size([2, 400]), torch.Size([2, 85]))

In [21]:
dls.show_batch(dataloaders=dls, max_n=2)

,text,target
0,"<s> Charges and counter charges flew in the Lok Sabha on Monday over the issue of lynchings, with the Opposition parties targeting the Modi government and the ruling side asserting that it was the responsibility of the state governments to deal with such crimes.During the discussion, Congress MP Mallikarjun Kharge criticised the Prime Minister for not talking about the issue enough and listed the various incidents of lynchings across the country. BJP?s Hukmdev Narayan Yadav defended the government?s response and, referring to political killings in Kerala, questioned why only certain incidents were considered lynchings.In the Rajya Sabha, after initial uproar over the alleged kidnapping of MLAs by police in Gujarat, the house adjourned briefly before resuming Question Hour.Congress? Madhusudhan Mistry alleged that his party MLAs were being?kidnapped? and offered?Rs 10-15 crore? with a view to influence their vote in the August 8 elections to send three members from the state to the Rajya Sabha.Here are the highlights: 6:30 pm: Lok Sabha adjourned for the day.6:29 pm:?The more you are raising the issue, the veil of the opposition is getting unveiled. The more you try to defame us, the more the Opposition gets defames: Kiren Rijiju6:23 pm: Mother Teresa was not allowed to enter Arunachal Pradesh during Congress rule. If there has been one instance of persecution of Christians in Arunachal Pradesh after our rule, I?ll resign: Kiren Rijiju6:13 pm: Mob lynching is the worst form of crime, it should be condemned universally. Why do you raise selective issue. Previously it was intolerance, then award-wapsi, freedom of expression: Rijiju. 6:08 pm: Do you want the PM should break the federal structure and take over state machinery: Rij</s>","Minister of State for Home Affairs, Kiren Rijiju while discussing the recent incidents of mob lynching in Parliament said, ""Do you want the PM should break the federal structure and take over state machinery?"" The statement which led to an uproar in the Parliament came after Congress MP Mallikarjun Kharge criticised the PM for not talking about the issue enough."
1,"<s> \n\n8.11am BST\n08:11\n\nThat brings our coverage of the AHRC?s landmark survey to a close.Here are the report?s main points:Submissions from survivors themselves told us how:Universities have promised to implement the report?s recommendations, but given how vocal advocates have been of past?institutional failures?, the strength of their response and the timeline of implementation will be something to keep an eye on.Updated\nat 8.16am BST\n\n\n8.00am BST\n08:00\n\nHere?s a final submission from someone who used to work as a residential tutor, who contacted Guardian Australia anonymously to share their story:I was working in one of the residential colleges as a member of the staff, and was supporting a student through a case of sexual assault. It was escalated to the university and an internal investigation was conducted (I was not involved in the investigation, but sat in during the interviews). Basically, it was a case of the student?s word against the perpetrator?s and therefore the university concluded that there was not enough evidence to take any action. Based on our experience and from the testimonies given by both sides, my colleagues at the college who were involved in supporting the student and I knew that the victim?s account is more likely to be true than not, but we could not do anything without proof. And because the report came back without any conclusive result, we could not take any official action against the perpetrator. It was frustrating and I felt that we had truly let the victim down. To make it worse, the victim and the alleged perpetrator had to remain staying in the same college till the end of the year.Updated\nat 8.03am BST\n\n\n7.54am BST\n07:54\n\nSome personal stories from Twitter that have been shared with my colleague Calla Wahlquist. All have been published with their permission. As 

## Training

In [22]:
seq2seq_metrics = {
        'rouge': {
            'compute_kwargs': { 'rouge_types': ["rouge1", "rouge2", "rougeL"], 'use_stemmer': True },
            'returns': ["rouge1", "rouge2", "rougeL"]
        },
        'bertscore': {
            'compute_kwargs': { 'lang': 'en' },
            'returns': ["precision", "recall", "f1"]
        },
        'bleu': { 'returns': "bleu" },
    }

In [23]:
model = HF_BaseModelWrapper(hf_model)
learn_cbs = [HF_BaseModelCallback]
fit_cbs = [HF_Seq2SeqMetricsCallback(custom_metrics=seq2seq_metrics)]

learn = Learner(dls, 
                model,
                opt_func=ranger,
                loss_func=CrossEntropyLossFlat(),
                cbs=learn_cbs,
                splitter=partial(seq2seq_splitter, arch=hf_arch)).to_fp16()

learn.create_opt() 
learn.freeze()

Downloading:   0%|          | 0.00/2.16k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/2.92k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/2.48k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.55k [00:00<?, ?B/s]

In [24]:
# learn.lr_find()

In [25]:
# b = dls.one_batch()
# preds = learn.model(b[0])
# len(preds),preds[0], preds[1].shape

In [26]:
learn.fit_one_cycle(1,
                    lr_max=3e-4,
                    cbs=fit_cbs)

epoch,train_loss,valid_loss,rouge1,rouge2,rougeL,bertscore_precision,bertscore_recall,bertscore_f1,bleu,time
0,1.913993,1.431403,0.516038,0.288226,0.391999,0.896127,0.911524,0.903649,0.244639,03:39


Downloading:   0%|          | 0.00/482 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/878k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

In [27]:
learn.show_results(learner=learn, max_n=2)

,text,target,prediction
0,"\n\n3.00pm BST\n15:00\n\nWell that?s all from me, thanks for reading. I?ll leave you with our initial race report:\n\n2.56pm BST\n14:56\n\nSebastian Vettel throws his trophy into the air. He now leads Lewis Hamilton by 14 points in the drivers? championship.?It was a really difficult race,? says Vettel.?I had my hands full. The steering was off. I had to stay focused for the whole race.? Here?s the race classification:INITIAL CLASSIFICATION (LAP 71/71): A win on his 50th Ferrari start for Sebastian Vettel? #HungarianGP?? #F1 pic.twitter.com/FnGuEQBNeXUpdated\nat 2.57pm BST\n\n\n2.49pm BST\n14:49\n\nVettel celebrates on top of his Ferrari, punching the air before throwing himself into the mechanics in red. He will enjoy the world championship lead for the next four weeks.Updated\nat 2.56pm BST\n\n\n2.45pm BST\n14:45\n\nHamilton finishes fourth, then, with Verstappen fifth and Alonso a brilliant sixth? he finished with the fastest lap of the race, not bad in that McLaren Honda. \n\n2.44pm BST\n14:44\n\nVettel has Kimi Raikkonen looming large in his mirrors but holds on to win the race, with his team-mate second ahead of Lewis Hamilton in third? no, he allows Bottas back through right at the very last.Updated\nat 2.58pm BST\n\n\n2.42pm BST\n14:42\n\nLap 69/70 Mercedes haven?t been able to swap Hamilton and Bottas back, an unlikely bonus for the Briton and his championship hopes.\n\n2.40pm BST","Ferrari's Formula One racer Sebastian Vettel won the Hungarian Grand Prix title, finishing ahead of teammate Kimi Raikkonen and Mercedes' Valtteri Bottas. Mercedes' British driver Lewis Hamilton finished fourth in the race. With the win, Vettel extended his lead over rival Lewis Hamilton to 14 points in the drivers' championship table.","Sebastian Vettel won the Hungarian Grand Prix on his 50th Ferrari start. He now leads Lewis Hamilton by 14 points in the drivers' championship. Vettel's team-mate Kimi Raikkonen finished second with Lewis Hamilton in third. The German will enjoy the world championship lead for the next four weeks. Hamilton finished fourth, with Verstappen fifth and Fernando Alonso sixth."
1,"Rejecting the Congress? charge of poaching its MLAs in Gujarat, the BJP has said it had no role in the recent desertion by six of its legislators and wondered whether they were?purchasable?.It also said that the sole of aim of the Congress was to save its vice president Rahul Gandhi in the country and Ahmed Patel in Gujarat.Taking a dig at the Congress after it ferried its MLAs to Bengaluru to keep its flock together ahead of the August 8 Rajya Sabha polls in Gujarat, Union minister Prakash Javadekar said the lawmakers were?enjoying themselves? there while those of the BJP were busy providing relief to the flood-hit people in the state.The party shepherded 44 lawmakers on Friday night to the holiday retreat in Karnakata after a spate of resignation of its MLAs amid allegations that the ruling BJP in Gujarat were bribing and coercing them ahead of the August 8 vice-presidential election.?Gujarat Congress leaders are leveling allegations at the BJP in Bengaluru. It is like a thief attacking a cop. If their leaders are leaving them, it has nothing to do with us,? Javadekar told PTI on Sunday after the Congress attacked the saffron party.He noted that senior Congress leaders like Shankersinh Vaghela, a former chief minister and the leader of opposition, its chief whip Balwantsinh Rajput and its spokesperson Tejashreeben Patel have left the party and wondered if the opposition party was suggesting that they could be?bought?.?They are all strong leaders. Nobody can purchase them. The Congress should look inwards and think why such senior leaders are leaving it. The truth is that its only goal is to save Rahul Gandhi in the country and Ahmed Patel in Gujarat,? he said.Patel is seeking re-election to the Rajya Sabha from Gujarat but defection by a number of Congress MLAs has left","Rejecting Congress' claims of its G

In [28]:
test_article = """
The past 12 months have been the worst for aviation fatalities so far this decade - with the total of number of people killed if airline 
crashes reaching 1,050 even before the Air Asia plane vanished. Two incidents involving Malaysia Airlines planes - one over eastern Ukraine and the other in the Indian Ocean - led to the deaths of 537 people, while an Air Algerie crash in Mali killed 116 and TransAsia Airways crash in Taiwan killed a further 49 people. The remaining 456 fatalities were largely in incidents involving small commercial planes or private aircraft operating on behalf of companies, governments or organisations. Despite 2014 having the highest number of fatalities so far this decade, the total number of crashes was in fact the lowest since the first commercial jet airliner took off in 1949 - totalling just 111 across the whole world over the past 12 months. The all-time deadliest year for aviation was 1972 when a staggering 2,429 people were killed in a total of 55 plane crashes - including the crash of Aeroflot Flight 217, which killed 174 people in Russia, and Convair 990 Coronado, which claimed 155 lives in Spain. However this year's total death count of 1,212, including those presumed dead on board the missing Air Asia flight, marks a significant rise on the very low 265 fatalities in 2013 - which led to it being named the safest year in aviation since the end of the Second World War. Scroll down for videos. Deadly: The past 12 months have been the worst for aviation fatalities so far this decade - with the total of number of people killed if airline crashes reaching 1,158 even before the Air Asia plane (pictured) vanished. Fatal: Two incidents involving Malaysia Airlines planes - one over eastern Ukraine (pictured) and the other in the Indian Ocean - led to the deaths of 537 people. Surprising: Despite 2014 having the highest number of fatalities so far this decade, the total number of crashes was in fact the lowest since the first commercial jet airliner took off in 1949. 2014 has been a horrific year for Malaysia-based airlines, with 537 people dying on Malaysia Airlines planes, and a further 162 people missing and feared dead in this week's Air Asia incident. In total more than half the people killed in aviation incidents this year had been flying on board Malaysia-registered planes. In January a total of 12 people lost their lives in five separate incidents, while the same number of crashes in February killed 107. 
"""

In [29]:
outputs = learn.blurr_generate(test_article,
                               early_stopping=True,
                               num_beams=4,
                               num_return_sequences=3)

for idx, o in enumerate(outputs):
    print(f'=== Prediction {idx+1} ===\n{o}\n')

=== Prediction 1 ===
 The past 12 months have been the worst for aviation fatalities so far this decade with 1,158 people killed if airline crashes across the world, including 537 on board Malaysia Airlines planes. Despite 2014 having the highest number of fatalities, the total number of crashes was in fact the lowest since the first commercial jet airliner took off in 1949 - totalling just 111 across the whole world. The all-time deadliest year for aviation was 1972 when 2,429 people were killed in a total of 55 plane crashes, including the crash of Aeroflot Flight 217, which killed 174 people in Russia.

=== Prediction 2 ===
 The past 12 months have been the worst for aviation fatalities so far this decade with 1,158 people killed if airline crashes across the world, including 537 on board Malaysia Airlines planes. Despite 2014 having the highest number of fatalities, the total number of crashes was in fact the lowest since the first commercial jet airliner took off in 1949 - totalli

In [30]:
df_test = pd.read_csv('news_summary - test.csv', 
                       engine='python',
                       encoding='unicode_escape')

In [31]:
from tqdm import tqdm

for i, row in tqdm(df_test.iterrows()):
  temp = row['Text']
  df_test.at[i, 'pred'] = learn.blurr_generate(temp,
                                               early_stopping=True,
                                               num_beams=1,
                                               num_return_sequences=1)[0]

49it [01:52,  2.29s/it]


In [32]:
from datasets import load_dataset, load_metric

metric = load_metric("rouge")

In [33]:
results = metric.compute(predictions=df_test['pred'],
                         references=df_test['Summary'])
print(results)

{'rouge1': AggregateScore(low=Score(precision=0.4139021450712501, recall=0.48950128762116496, fmeasure=0.44742442395134224), mid=Score(precision=0.4452446504124026, recall=0.5243150626379718, fmeasure=0.47718485103663344), high=Score(precision=0.4768362289563184, recall=0.5596212312838647, fmeasure=0.5087055136709423)), 'rouge2': AggregateScore(low=Score(precision=0.19396010825251855, recall=0.22622479351403274, fmeasure=0.20760686686917584), mid=Score(precision=0.22867921123383728, recall=0.26717201826101944, fmeasure=0.244165213222259), high=Score(precision=0.2683815984086174, recall=0.31056649189801133, fmeasure=0.28395071369387476)), 'rougeL': AggregateScore(low=Score(precision=0.29202185883662457, recall=0.34467164595561056, fmeasure=0.31483701687328797), mid=Score(precision=0.323115519430363, recall=0.37865878806415537, fmeasure=0.3460093994688227), high=Score(precision=0.3565846860567897, recall=0.4137918115216059, fmeasure=0.37749551809738047)), 'rougeLsum': AggregateScore(low=

In [34]:
for key in ["rouge1", "rouge2", "rougeL"]:
    print(key, "low:", results[key].low.fmeasure)
    print(key, "mid:", results[key].mid.fmeasure)
    print(key, "high:", results[key].high.fmeasure, end="\n\n")

rouge1 low: 0.44742442395134224
rouge1 mid: 0.47718485103663344
rouge1 high: 0.5087055136709423

rouge2 low: 0.20760686686917584
rouge2 mid: 0.244165213222259
rouge2 high: 0.28395071369387476

rougeL low: 0.31483701687328797
rougeL mid: 0.3460093994688227
rougeL high: 0.37749551809738047



In [35]:
from nltk.translate.bleu_score import sentence_bleu

In [39]:
df_test.head()

,Summary,Text,pred
0,The Administration of Union Territory Daman and Diu has revoked its order that made it compulsory for women to tie rakhis to their male colleagues on the occasion of Rakshabandhan on August 7. The administration was forced to withdraw the decision within 24 hours of issuing the circular after it received flak from employees and was slammed on social media.,"The Daman and Diu administration on Wednesday withdrew a circular that asked women staff to tie rakhis on male colleagues after the order triggered a backlash from employees and was ripped apart on social media.The union territory?s administration was forced to retreat within 24 hours of issuing the circular that made it compulsory for its staff to celebrate Rakshabandhan at workplace.?It has been decided to celebrate the festival of Rakshabandhan on August 7. In this connection, all offices/ departments shall remain open and celebrate the festival collectively at a suitable time wherein a...",The Daman and Diu administration on Wednesday withdrew a circular that asked women staff to tie rakhis on male colleagues for the festival of Rakshabandhan on August 7. The order had been issued on August 1 by the UT's department of personnel and administrative reforms. The circular was withdrawn through a one-line order issued late in the evening by the department of Personnel and Administrative Reforms.
1,"Malaika Arora slammed an Instagram user who trolled her for ""divorcing a rich man"" and ""having fun with the alimony"". ""Her life now is all about wearing short clothes, going to gym or salon, enjoying vacation[s],"" the user commented. Malaika responded, ""You certainly got to get your damn facts right before spewing sh*t on me...when you know nothing about me.""","From her special numbers to TV?appearances, Bollywood actor Malaika Arora Khan has managed to carve her own identity. The actor, who made her debut in the Hindi film industry with the blockbuster debut opposite Shah Rukh Khan in Chaiyya Chaiyya from Dil Se (1998), is still remembered for the song. However, for trolls, she is a woman first and what matters right now is that she divorced a ?rich man?. On Wednesday, Malaika Arora shared a gorgeous picture of herself on Instagram and a follower decided to troll her for using her ?alumni? (read alimony) money to wear ?short clothes and going t...","A follower of Malaika Arora Khan, who has been taunting her for using her alimony money to buy short clothes and go to the gym, tweeted that she should get a course in feminism. Malaikas replied that she is a successful woman who has played all the roles traditionally prescribed for a woman. She added that she did not need an alimony to buy clothes (short or not, her choice) and enjoy her life."
2,"The Indira Gandhi Institute of Medical Sciences (IGIMS) in Patna on Thursday made corrections in its Marital Declaration Form by changing 'Virgin' option to 'Unmarried'. Earlier, Bihar Health Minister defined virgin as being an unmarried woman and did not consider the term objectionable. The institute, however, faced strong backlash for asking new recruits to declare their virginity in the form.","The Indira Gandhi Institute of Medical Sciences (IGIMS) in Patna amended its marital declaration form on Thursday, replacing the word ?virgin? with ?unmarried? after controversy.Until now, new recruits to the super-specialty medical institute in the state capital were required to declare if they were bachelors, widowers or virgins.IGIMS medical superintendent Dr Manish Mandal said institute director Dr NR Biswas held a meeting on Thursday morning before directing that the word ?virgin? on the marital declaration form be immediately replaced with ?unmarried?. Dr Biswas had just returned aft...","The Indira Gandhi Institute of Medical Sciences in Patna has replaced the word ?virgin? on its marital declaration form with ?unmarried? after controversy over the word 'virgin' on Thursday. The word 'Virgin' had been used to as

In [37]:
bleu = []
for i, row in tqdm(df_test.iterrows()):
  bleu.append(sentence_bleu([row['Summary'].split()], row['pred'].split()))

0it [00:00, ?it/s]/usr/local/lib/python3.7/dist-packages/nltk/translate/bleu_score.py:490: UserWarning: 
Corpus/Sentence contains 0 counts of 2-gram overlaps.
BLEU scores might be undesirable; use SmoothingFunction().
  warnings.warn(_msg)
49it [00:09,  5.05it/s]


In [38]:
print('bleu   :',   np.mean(bleu))

bleu   : 0.5322325279387415


In [41]:
df_test.to_csv('bart.csv', index=False)

False